# 06 Encoding

This notebook is a starting point for encoding categorical variables in the cleaned ElectraHub dataset.

Suggested flow:
- load the cleaned dataset
- review categorical columns
- apply encoding to selected features
- verify the transformed output

In [2]:
from pathlib import Path

import pandas as pd

dataset_path = Path("../data/processed/ElectraHub_Data_cleaned.csv")
df = pd.read_csv(dataset_path)

print(f"Dataset path: {dataset_path}")
print(f"Rows loaded: {len(df)}")
print(f"Columns loaded: {len(df.columns)}")

Dataset path: ../data/processed/ElectraHub_Data_cleaned.csv
Rows loaded: 3000
Columns loaded: 16


In [3]:
df.head()

  Region Product_Category  ... Popularity     Sales
0  South           Tablet  ...   Moderate  12265.79
1   West           Tablet  ...       High  20426.45
2   West           Mobile  ...       High  29827.18
3   East           Tablet  ...   Moderate  28211.95
4  North           Mobile  ...       High  29794.80

[5 rows x 16 columns]

In [4]:
categorical_columns = df.select_dtypes(exclude="number").columns.tolist()
print("Categorical columns:", categorical_columns)

Categorical columns: ['Region', 'Product_Category', 'Campaign_Type', 'Popularity']


## Encoding

Use this section to convert categorical variables into model-ready numeric features after splitting the data.

In [6]:
train_raw = df.sample(frac=0.8, random_state=42)
test_raw = df.drop(train_raw.index)

print(f"Train rows: {len(train_raw)}")
print(f"Test rows: {len(test_raw)}")
print(f"Train shape before encoding: {train_raw.shape}")
print(f"Test shape before encoding: {test_raw.shape}")
train_raw.head()

Train rows: 2400
Test rows: 600
Train shape before encoding: (2400, 16)
Test shape before encoding: (600, 16)


     Region Product_Category  ... Popularity     Sales
1801   West           Mobile  ...   Very Low   7812.55
1190  North           Tablet  ...       High  20575.14
1817   West           Tablet  ...        Low  18559.52
251   South           Mobile  ...  Very High  31278.53
2505  North           Mobile  ...   Moderate  25305.45

[5 rows x 16 columns]

## Train/Test Split

Set aside the test set here using an 80/20 split with `random_state=42`.

In [8]:
ordinal_mapping = {
    "Very Low": 0,
    "Low": 1,
    "Moderate": 2,
    "High": 3,
    "Very High": 4,
}

nominal_columns = [column for column in categorical_columns if column != "Popularity"]

train_nominal_dummies = pd.get_dummies(train_raw[nominal_columns], drop_first=True)
train_nominal_columns = train_nominal_dummies.columns.tolist()
test_nominal_dummies = pd.get_dummies(test_raw[nominal_columns], drop_first=True).reindex(columns=train_nominal_columns, fill_value=0)

train_encoded = pd.concat([train_raw.drop(columns=nominal_columns), train_nominal_dummies], axis=1)
test_encoded = pd.concat([test_raw.drop(columns=nominal_columns), test_nominal_dummies], axis=1)

train_encoded["Popularity"] = train_raw["Popularity"].map(ordinal_mapping)
test_encoded["Popularity"] = test_raw["Popularity"].map(ordinal_mapping)

final_columns = train_encoded.columns.tolist()
train_encoded = train_encoded[final_columns]
test_encoded = test_encoded[final_columns]

train_path = Path("../data/processed/ElectraHub_train_encoded.csv")
test_path = Path("../data/processed/ElectraHub_test_encoded.csv")

train_encoded.to_csv(train_path, index=False)
test_encoded.to_csv(test_path, index=False)

print(f"Train rows: {len(train_encoded)}")
print(f"Test rows: {len(test_encoded)}")
print(f"Saved train set to: {train_path}")
print(f"Saved test set to: {test_path}")
print("Final columns:")
print(final_columns)

Train rows: 2400
Test rows: 600
Saved train set to: ../data/processed/ElectraHub_train_encoded.csv
Saved test set to: ../data/processed/ElectraHub_test_encoded.csv
Final columns:
['Product_Age_Months', 'Product_Price', 'Competitor_Price_Index', 'Advertising_Expenditure', 'Discount_Percentage', 'Campaign_Engagement_Score', 'Inventory_Level', 'Num_Reviews', 'Avg_Customer_Rating', 'Return_Rate', 'Length_Product_Description', 'Popularity', 'Sales', 'Region_North', 'Region_South', 'Region_West', 'Product_Category_Tablet', 'Campaign_Type_Influencer', 'Campaign_Type_Search Ad', 'Campaign_Type_Social Media']


- If someone just guessed the average Sales of $26,253 for every single product, the RMSE would be $7,053 and the MAE would be $5,708. That requires zero modelling. 
- Any model that barely beats these numbers is not adding much value.
- For simple LR models — one variable each — R² above 0.50 is a reasonable target. For the full
- multiple regression model, the expectation is higher: R² above 0.85, RMSE well below $7,053, ideally below $3,000.

## Findings

Document any encoding decisions or notes here.